# Equity curves and metrics, train and test

**What the professor asks** ([resources/instructions.pdf](../resources/instructions.pdf), "The Evaluation Framework"): evaluate the strategy with suitable metrics, **on both the train and the test**, and compare it with a **benchmark**, justifying the choice. Among the metrics: cumulative return drawn as an **equity curve starting at 100, with the benchmark on the same chart**, annualised return, annualised volatility, Sharpe ratio (stating the risk-free rate), maximum drawdown, profit factor.

**Where the returns come from.** From the +1 / 0 / -1 signal per stock of [08_signal.ipynb](08_signal.ipynb), with the professor's formula: stock return x signal. They are the daily returns saved in `Results/signal_returns.csv` (train) and `Results/test/signal_returns.csv` (test). They match those of the trades to within 0.3%, but Sharpe and drawdown here can differ slightly from those of Steps 6 and 7, which are computed from the trades. This notebook only reads saved files: it does not redo the test.

**The versions.** **A** (no agent), **B** and **C** of `finbert_episode`, the ones taken to the test.

**The benchmarks and why.**
- **SPY bought and held** (the ETF on the S&P 500): it is the US stock market the 30 shares come from, the comparison suggested by the professor. It tells whether the strategy earns more or less than the stock market and whether it depends on it.
- **Version A**: it is the strategy without the agent. It is the right comparison to say whether the news agent helps, which is the question of the project.
- We do not use the 30 stocks bought and held: the strategy buys and sells at the same time and is almost always out of the market, so that comparison would say little.

## The computation assumptions

- **Capital and strategy curve.** 300,000 dollars (20,000 per pair), without reinvestment, as in the backtest: the return of a day is the profit of the day divided by 300,000, and the curve is 100 x (1 + sum of the daily returns). Days without positions count as 0.
- **SPY curve.** Bought at the close of the day before the start and held: 100 x product of (1 + daily return). Prices adjusted for dividends, from Alpaca.
- **Annualised return**, the same for all: (1 + cumulative return) to the power 252 / days, minus 1. Over 64 days (train) and 32 (test) it is not very reliable: it scales a few months to a year.
- **Annualised volatility:** standard deviation of the daily returns x square root of 252.
- **Sharpe:** mean of the daily returns / standard deviation x square root of 252, with a **risk-free rate of 0**. Choice: the strategy closes everything every evening and the capital is not invested overnight; stated here and in the report.
- **Maximum drawdown:** the largest loss of the curve from a previous peak, in percent.
- **Profit factor:** sum of the gains of the positive days divided by the sum of the losses of the negative days (on days, because the returns come from the signal; the per-trade one is in Steps 6 and 7).
- **Costs:** 0 (main, the professor's indication) and 5 basis points per operation as a comparison. An operation is each change by 1 of the signal of a stock: opening a trade is 2 operations (two stocks), closing it another 2, so 4 per trade as in the backtest. Cost of an operation: 5 basis points of 10,000 dollars, i.e. 5 dollars.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # project folder: makes src importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import RESULTS, RESULTS_TEST, SPY_DAILY, SPY_TEST, VARIANTS
from src.backtest import DAYS_PER_YEAR, NOTIONAL_LEG
from src.agent_variants.comparison import beta_alpha

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
CAPITAL = 300_000
PERIODS = ("train", "test")
ret = {"train": pd.read_csv(RESULTS / "signal_returns.csv", index_col="day", parse_dates=True),
       "test": pd.read_csv(RESULTS_TEST / "signal_returns.csv", index_col="day", parse_dates=True)}
signals = {("train", "A"): RESULTS / "signal_A.parquet", ("train", "B"): VARIANTS / "finbert_episode/signal_B.parquet",
           ("train", "C"): VARIANTS / "finbert_episode/signal_C.parquet"} | {("test", v): RESULTS_TEST / f"signal_{v}.parquet" for v in "ABC"}
spy = {"train": pd.read_parquet(SPY_DAILY), "test": pd.read_parquet(SPY_TEST)}
for per in PERIODS:
    close = spy[per].set_index("date").close
    ret[per]["SPY"] = close.pct_change().reindex(ret[per].index)    # the first day uses the close of the day before
    assert ret[per].notna().all().all(), f"SPY is missing on some day of the {per}"
    print(f"{per}: {len(ret[per])} days, from {ret[per].index.min():%d/%m/%Y} to {ret[per].index.max():%d/%m/%Y}")

## 1. The costs from the signal

For the 5-basis-point scenario: for each day the operations are counted, i.e. the changes of the signal of each stock, including the closing at the end of the day. The check: the operations are exactly 4 per trade.

In [ ]:
def operations_per_day(s: pd.DataFrame) -> pd.Series:
    """Number of operations (changes by 1 of the signal of a stock) per day. Every day starts at 0 and ends at 0."""
    def one_day(x):
        return x.diff().fillna(x).abs().to_numpy().sum() + x.iloc[-1].abs().sum()
    return s.groupby(s.index.normalize()).apply(one_day)


ops = {}
for (per, v), path in signals.items():
    ops[(per, v)] = operations_per_day(pd.read_parquet(path)).reindex(ret[per].index, fill_value=0)
pd.DataFrame({"operations": {k: int(o.sum()) for k, o in ops.items()}, "trade": {k: o.sum() / 4 for k, o in ops.items()}}).rename_axis(["period", "version"])

## 2. Equity curves from 100

A, B and C (cost 0) and SPY on the same chart, for the train and for the test. All the curves are worth 100 at the close of the trading day before the period (15 June for the train, 16 September for the test).

In [ ]:
START = {per: spy[per].date[spy[per].date < ret[per].index.min()].max() for per in PERIODS}   # trading day before the period: the curve is worth 100


def curve(r: pd.Series, name: str, per: str) -> pd.Series:
    """Curve from 100 at the close of the day before the period: the strategy sums the returns (fixed capital), SPY compounds them."""
    eq = 100 * (1 + r).cumprod() if name == "SPY" else 100 * (1 + r.cumsum())
    return pd.concat([pd.Series([100.0], index=[START[per]]), eq])


STYLE = {"A": ("black", "A, no agent"), "B": ("tab:blue", "B, finbert_episode blocks"),
         "C": ("tab:orange", "C, finbert_episode blocks or follows"), "SPY": ("tab:green", "SPY bought and held (benchmark)")}
for per, path in (("train", RESULTS / "equity_curve_100.png"), ("test", RESULTS_TEST / "equity_curve_100.png")):
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for name, (colour, label) in STYLE.items():
        ax.plot(curve(ret[per][name], name, per), color=colour, lw=2 if name in ("A", "SPY") else 1.4, label=label)
    ax.axhline(100, color="grey", lw=0.6)
    ax.set_ylabel("value (start = 100)")
    ax.set_title(f"{per.capitalize()}, {ret[per].index.min():%d/%m} - {ret[per].index.max():%d/%m/%Y}: 15 pairs, cost 0")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.show()

## 3. Metrics, train and test

One row per period, cost and version; SPY has no costs. `trade` is the number of trades (operations / 4).

In [ ]:
def metrics(r: pd.Series, name: str, per: str) -> dict:
    eq = curve(r, name, per)
    cum = eq.iloc[-1] / 100 - 1
    return dict(cumulative_return_pct=100 * cum,
                annual_return_pct=100 * ((1 + cum) ** (DAYS_PER_YEAR / len(r)) - 1),
                annual_volatility_pct=100 * r.std() * np.sqrt(DAYS_PER_YEAR),
                sharpe=r.mean() / r.std() * np.sqrt(DAYS_PER_YEAR),
                max_drawdown_pct=100 * (eq / eq.cummax() - 1).min(),
                profit_factor=r[r > 0].sum() / -r[r < 0].sum())


rows = []
for per in PERIODS:
    for cost in (0, 5):
        for v in "ABC":
            r = ret[per][v] - ops[(per, v)] * cost / 1e4 * NOTIONAL_LEG / CAPITAL
            rows.append(dict(period=per, cost_bps=cost, version=v, trade=int(ops[(per, v)].sum() / 4)) | metrics(r, v, per))
    rows.append(dict(period=per, cost_bps=0, version="SPY", trade=np.nan) | metrics(ret[per]["SPY"], "SPY", per))
table = pd.DataFrame(rows).set_index(["period", "cost_bps", "version"])
table

## 4. Does the strategy depend on the market?

For the "market-neutral" bonus point the professor looks, on the test, at: beta against the S&P 500 close to zero, low correlation with the index and a positive mean return on the days when the index falls. Here for A, B and C (cost 0), on train and test. The beta comes from the regression of the daily returns on those of SPY.

In [ ]:
rows = []
for per in PERIODS:
    spy_r = ret[per]["SPY"]
    down = spy_r < 0
    for v in "ABC":
        r = ret[per][v]
        ba = beta_alpha(r, spy_r)
        rows.append(dict(period=per, version=v, beta=ba["beta"], t_beta=ba["t_beta"], correlation=r.corr(spy_r),
                         spy_down_days=int(down.sum()), mean_return_spy_down_days_bps=1e4 * r[down].mean(),
                         share_positive_spy_down_days=(r[down] > 0).mean(), spy_mean_return_down_days_bps=1e4 * spy_r[down].mean()))
neutrality = pd.DataFrame(rows).set_index(["period", "version"])
neutrality

## 5. Is the Sharpe ratio of the strategy really higher than that of SPY?

**What the professor asks** ("superior risk-adjusted return" bonus point): on the test the strategy must have a higher Sharpe ratio and a lower drawdown than the benchmark, and the Sharpe difference must be supported by a statistical test, for example a bootstrap.

**The test (bootstrap on the days).** We take the 32 days of the test, each with the return of the strategy and that of SPY. We draw 32 days at random with replacement, **the same days for the strategy and for SPY**, and compute the two Sharpe ratios with the formula of section 3 and their difference. This is repeated 10,000 times (seed 0). If the Sharpe of the strategy were higher only by chance, in many draws the difference would be negative.

**Choices** (decided before computing): test only, cost 0, versions A, B and C against SPY, days drawn one by one (as in the bootstrap of Step 7).

**Reading rule, written before computing:** the difference is **supported by the test** if the 95% interval of the difference does not contain zero.

Next to it are the maximum drawdowns of section 3, the second condition of the bonus.

In [ ]:
def sharpe_rows(x: np.ndarray) -> np.ndarray:
    """Annualised Sharpe of each row (one draw), risk-free rate 0, as in section 3."""
    return x.mean(axis=1) / x.std(axis=1, ddof=1) * np.sqrt(DAYS_PER_YEAR)


t = ret["test"]
draws = np.random.default_rng(0).integers(0, len(t), size=(10_000, len(t)))   # 10,000 draws of 32 days, the same for all the columns
sharpe_spy_boot = sharpe_rows(t["SPY"].to_numpy()[draws])
rows = []
for v in "ABC":
    diff = sharpe_rows(t[v].to_numpy()[draws]) - sharpe_spy_boot
    low, high = np.percentile(diff, [2.5, 97.5])
    rows.append(dict(version=v, sharpe_strategy=table.loc[("test", 0, v), "sharpe"], sharpe_spy=table.loc[("test", 0, "SPY"), "sharpe"],
                     difference=table.loc[("test", 0, v), "sharpe"] - table.loc[("test", 0, "SPY"), "sharpe"],
                     ci95_low=low, ci95_high=high, share_strategy_below_spy=(diff < 0).mean(), supported=bool(low > 0),
                     max_drawdown_strategy_pct=table.loc[("test", 0, v), "max_drawdown_pct"],
                     max_drawdown_spy_pct=table.loc[("test", 0, "SPY"), "max_drawdown_pct"]))
sharpe_spy = pd.DataFrame(rows).set_index("version")
sharpe_spy

## 6. Saving

`Results/metrics_train_test.csv`, `Results/market_neutrality.csv`, `Results/test/sharpe_vs_spy.csv` and the two charts (`Results/equity_curve_100.png`, `Results/test/equity_curve_100.png`).

In [ ]:
table.to_csv(RESULTS / "metrics_train_test.csv")
neutrality.to_csv(RESULTS / "market_neutrality.csv")
sharpe_spy.to_csv(RESULTS_TEST / "sharpe_vs_spy.csv")
for p in (RESULTS / "metrics_train_test.csv", RESULTS / "market_neutrality.csv", RESULTS_TEST / "sharpe_vs_spy.csv", RESULTS / "equity_curve_100.png", RESULTS_TEST / "equity_curve_100.png"):
    print(p.relative_to(RESULTS.parent), p.exists())

## 7. What the results tell us

Returns from the signal, cost 0 unless stated. A without the agent, B and C with `finbert_episode`.

| | A | B | C | SPY |
|---|---|---|---|---|
| **Train:** cumulative return | +2.42% | +2.32% | +2.20% | +0.15% |
| annualised return | +9.9% | +9.5% | +8.9% | +0.6% |
| annualised volatility | 1.2% | 1.1% | 1.2% | 11.7% |
| Sharpe (risk-free rate 0) | 8.2 | 8.2 | 7.5 | 0.1 |
| maximum drawdown | -0.14% | -0.14% | -0.14% | -3.4% |
| profit factor (on days) | 3.9 | 3.9 | 3.5 | 1.0 |
| **Test:** cumulative return | +0.83% | +0.76% | +0.41% | +0.47% |
| annualised return | +6.7% | +6.2% | +3.3% | +3.8% |
| annualised volatility | 1.3% | 1.3% | 1.2% | 11.4% |
| Sharpe (risk-free rate 0) | 4.9 | 4.7 | 2.8 | 0.4 |
| maximum drawdown | -0.17% | -0.19% | -0.21% | -3.0% |
| profit factor (on days) | 2.5 | 2.3 | 1.6 | 1.1 |

**The agent does not improve the strategy, not even with these metrics.** In both periods B is slightly below A and C further below, on every metric except volatility, which is the same for all. These are the same results as Steps 6 and 7, seen as an equity curve.
- **B** is almost the same as A: on the test it blocks only 7 trades out of 500. Return +0.76% against +0.83%, Sharpe 4.7 against 4.9.
- **C** is the worst. On the test it returns +0.41%, less even than SPY (+0.47%), though with much less risk; Sharpe 2.8, drawdown -0.21%.

**Against SPY, at cost 0, all three versions have much less risk**: volatility 1.1-1.3% a year against 11-12%, drawdown below 0.25% against 3%. On the return A and B do better than SPY in both periods; C does on the train, not on the test.

**To be read with caution.**
- The volatility is very low because the capital is invested only during the day, in a few trades at a time, and never overnight. For this reason the Sharpe ratio is very high: it must not be compared with that of a fund.
- The annualised figures scale 64 days (train) and 32 (test) to a year. In these periods SPY is almost flat: +0.15% on the train and +0.47% on the test.
- **With 5 basis points of cost all versions lose:** on the test A -2.5%, B -2.5%, C -2.9%; on the train A -4.1%, B -4.1%, C -4.3%. The gross profit per trade is smaller than a realistic cost. Cost 0 is the professor's indication.

**Consistency with notebooks 06 and 07.** From the signal the Sharpe of A on the test is 4.93; from the trades it was 4.92. The drawdown at cost 5 matches (-2.65% on the test, -4.07% on the train).

**Dependence on the market** (section 4, "market-neutral" bonus point), on the test:

| | A | B | C |
|---|---|---|---|
| beta against SPY (t statistic) | -0.04 (-2.2) | -0.04 (-2.1) | -0.04 (-2.3) |
| correlation with SPY | -0.37 | -0.36 | -0.39 |
| mean profit on the 16 days when SPY falls | +4.0 bps | +3.6 bps | +2.8 bps |
| share of those days in profit | 62% | 62% | 56% |

- **Beta close to zero** for all three, on the train (0.01) and on the test (-0.04).
- **On the days when SPY falls** (on the test -55 basis points on average) all three earn on average, C less than the others. On the train: +2.7 bps for A, +2.6 for B, +2.6 for C, positive on 69% of the days.
- **Correlation with SPY:** low on the train (0.07-0.08), more marked and negative on the test (from -0.36 to -0.39). On the test the beta, though small, can be told apart from zero: the strategy tends to earn a bit more when the stock market falls.
- **Small returns in absolute terms:** less than 1% in six weeks. The professor asks for "relevant and consistent" returns; whether these are must be assessed.

**The Sharpe higher than SPY's is not supported by the statistical test** (section 5, "superior risk-adjusted return" bonus point), on the test at cost 0:

| | A | B | C |
|---|---|---|---|
| Sharpe of the strategy minus Sharpe of SPY | +4.6 | +4.4 | +2.4 |
| 95% interval of the difference | from -4.6 to +12.5 | from -4.8 to +12.4 | from -7.3 to +10.6 |
| share of draws with the strategy below SPY | 15% | 16% | 31% |
| maximum drawdown (SPY -3.0%) | -0.17% | -0.19% | -0.21% |

- **The drawdown is lower** for all three: the second condition of the bonus holds.
- **The Sharpe difference is not supported:** the interval contains zero for A, B and C, so with the rule written beforehand the bonus cannot be claimed. The main reason is SPY: over 32 days its Sharpe is very uncertain (its volatility is ten times that of the strategy), and it is enough to draw a few good market days to bring it above that of the strategy.
- With 5 basis points of cost the question does not arise: the Sharpe of the strategy is negative.
- The professor asks for a **long and challenging** test period; our 32 days, with an almost flat market, are not. With a longer period the interval would narrow, but we do not know in which direction.

**Limits.**
- Train and test are short and in both SPY moves little: a single kind of market.
- Risk-free rate 0: with the yield of government bonds the Sharpe ratio would be lower.
- The bootstrap draws the days one by one: it does not account for nearby days being similar.